# Playground — Subspace concept pipeline

Alternative to the frame representation: a concept is the top-`r` SVD subspace of the
pooled whitened+centered unembedding features of every token across every lemma.

New API (`subspace/`):

- `SubspaceUnembeddingRepresentation.from_model_id(id, rank=...)` — reuses the FRH
  model / unembedding / WordNet loaders.
- `sur.get_concept_cached(synset, min_lemmas, max_tokens)` → `ConceptSubspace` (basis
  cached under `cache/subspaces/`).
- `sur.project(text, concept)` — per-position alignment `‖Bᵀh‖`.
- `sur.expression(text, concept)` — mean energy fraction in [0, 1].
- `sur.generate_with_topk_guide(text, guide, k, steps, fluency_weight=0.0)` —
  subspace-guided generation; `fluency_weight` = λ is the fluency dial.
- `ConceptSubspace.blend([c1, c2, ...])` — merge concepts into one guide
  (Family-1 composition).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames`/`subspace` import

In [ ]:
import torch

from subspace import SubspaceUnembeddingRepresentation

In [ ]:
sur = SubspaceUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    rank=8,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3

## Build a couple of concept subspaces

In [ ]:
dog = sur.get_concept_cached("dog.n.01", MIN_LEMMAS, MAX_TOKENS)
war = sur.get_concept_cached("war.n.01", MIN_LEMMAS, MAX_TOKENS)
print(dog)
print(war)

# sanity: the basis columns are orthonormal (up to the kept rank)
gram = dog.basis.mT.float() @ dog.basis.float()
print(
    "BᵀB ≈ I:", torch.allclose(gram, torch.eye(dog.rank, device=gram.device), atol=1e-4)
)

## Projection: concept-relevant text scores higher

In [ ]:
relevant = "The puppy barked and wagged its tail at the hound"
unrelated = "The committee reviewed the quarterly budget report"

print("expression(relevant, dog):", sur.expression(relevant, dog).tolist())
print("expression(unrelated, dog):", sur.expression(unrelated, dog).tolist())

## Subspace-guided generation vs. an unguided baseline

In [ ]:
prompt = "Tell me about my day."

guided, probe = sur.generate_with_topk_guide(prompt, guide=dog, k=4, steps=24)
baseline = sur.model.decode(sur.model.generate([prompt], max_new_tokens=24))

print("guided:  ", guided[0])
print("baseline:", baseline[0])
print("final probe:", probe[..., -1].tolist())

## Blend composition (Family 1) + the fluency dial (λ)

`ConceptSubspace.blend([...])` merges several concepts into **one** subspace (the
principal subspace of the stacked bases), so multi-concept steering reuses the
same single-guide loop. `generate_with_topk_guide(..., fluency_weight=λ)` then
trades steering strength against the model's own fluency
(`cumulative alignment + λ · cumulative token log-prob`); `λ = 0` is the original
path unchanged.

In [ ]:
from subspace import ConceptSubspace

# Blend dog + war into a single guide that steers toward both at once.
blended = ConceptSubspace.blend([dog, war])
print("blended:", blended, "| synsets:", blended.synset)

prompt = "Tell me about my day."
for lam in (0.0, 0.4):
    texts, probe = sur.generate_with_topk_guide(
        prompt, guide=blended, k=4, steps=24, fluency_weight=lam
    )
    print(f"\nλ={lam} | final probe={probe[..., -1].item():.3f}")
    print(" ", texts[0])